In [ ]:
REPO_URL = "https://github.com/huyvanzzz/finetune-InternVL2.git"
TARGET_BRANCH = "feature/trajectory-pretrain-qformer"
PROJECT_DIR = "/kaggle/working/finetune-InternVL2"
CONFIG_PATH = "internvl_pretrain_config_traj_cls.yaml"
RESUME_CHECKPOINT = ""  # local folder or HF repo id; leave empty to train from scratch


In [ ]:
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "fetch", "origin", TARGET_BRANCH], check=True)
subprocess.run(["git", "checkout", "-B", TARGET_BRANCH, f"origin/{TARGET_BRANCH}"], check=True)
print("Current repo:", os.getcwd())
print("Current branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Current commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
!pip uninstall -y torch torchvision torchaudio triton flash-attn torchao peft || true
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1
!pip install -q --no-cache-dir bitsandbytes transformers==4.46.2 datasets accelerate timm evaluate rouge_score scikit-learn safetensors huggingface_hub sentencepiece mlflow pyyaml pillow tqdm peft==0.18.1


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    x = torch.zeros(1, device='cuda', dtype=torch.float16)
    print('cuda fp16 tensor ok:', x.dtype)
!python -m py_compile train_pretrain.py pretrain_dataset.py qformer_bridge.py trajectory_branch.py scripts/prepare_qformer.py


In [ ]:
import subprocess
subprocess.run(["python", "build_frame_index.py"], input="n\n", text=True, check=True)


In [ ]:
!python scripts/prepare_qformer.py --config {CONFIG_PATH}


In [ ]:
import subprocess
cmd = ["python", "train_pretrain.py", "--config", CONFIG_PATH]
if RESUME_CHECKPOINT:
    cmd += ["--checkpoint", RESUME_CHECKPOINT]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
